# Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import OxfordIIITPet
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device: ' + str(device))


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ------------------------- -------------- 2.4/3.7 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 10.9 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.
Using device: cpu


# Data

In [3]:
# Since params inside ResNet18 is based of these values, we should use them and not the ones for oxford-IIIT
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# Again 224 is needed since its what ResNet18 wants as input
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

# For binary (0,1)
train_set_binary = OxfordIIITPet(root='data', split='trainval', target_types='binary-category', transform=train_transform, download=True)
test_set_binary  = OxfordIIITPet(root='data', split='test', target_types='binary-category', transform=test_transform,  download=True)

train_loader_binary = DataLoader(train_set_binary, batch_size=50, shuffle=True,  num_workers=4) # Maybe batch size or/and num_workers should change value
test_loader_binary  = DataLoader(test_set_binary, batch_size=50, shuffle=False, num_workers=4)

# For full 37 (0-36)
train_set_full = OxfordIIITPet(root='data', split='trainval', target_types='category', transform=train_transform, download=True)
test_set_full  = OxfordIIITPet(root='data', split='test', target_types='category', transform=test_transform,  download=True)

train_loader_full = DataLoader(train_set_full, batch_size=50, shuffle=True,  num_workers=4)
test_loader_full  = DataLoader(test_set_full, batch_size=50, shuffle=False, num_workers=4)



print(f'Train size for binary: {len(train_set_binary)}, Test size: {len(test_set_binary)}')
print(f'Train size for full: {len(train_set_full)}, Test size: {len(test_set_full)}')

100.0%
100.0%


Train size for binary: 3680, Test size: 3669
Train size for full: 3680, Test size: 3669


# Training for Binary Classification

In [4]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2).to(device)

1.7%

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\simon/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


In [5]:
# Training for binary classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_binary:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_binary)
    train_acc  = correct / len(train_set_binary)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_binary:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0 = cat, 1 = dog for binary
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_binary)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')

Epoch [01/2]  Loss: 0.2531  Train Acc: 0.8970  Test Acc: 0.9695
Epoch [02/2]  Loss: 0.1050  Train Acc: 0.9652  Test Acc: 0.9785


In [6]:
print('Final test accuracy for binary classification: ' + str(test_acc*100) + '%') # I get 97.7% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)

Final test accuracy for binary classification: 97.84682474788771%


# Training for Full Classification

In [7]:
model = torchvision.models.resnet18(weights='IMAGENET1K_V1') # Only choice (of weights param)?

# Matching the tutorial
for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 37).to(device)

In [ ]:
# Training for full classification
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3) # Maybe lr should change for greater than 99%

num_epochs = 2

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0

    for images, labels in train_loader_full:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = running_loss / len(train_set_full)
    train_acc  = correct / len(train_set_full)


    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader_full:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            # 0-36 depending on breed
            correct += (outputs.argmax(1) == labels).sum().item()

    test_acc = correct / len(test_set_full)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}]  Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f}  Test Acc: {test_acc:.4f}')

Epoch [01/2]  Loss: 2.0535  Train Acc: 0.5516  Test Acc: 0.7961
Epoch [02/2]  Loss: 0.7685  Train Acc: 0.8636  Test Acc: 0.8444


In [9]:
print('Final test accuracy for full classification: ' + str(test_acc*100) + '%') # I get 84.4% after 2 epochs in vscode (~3min per epoch, takes a lot longer in colab)

Final test accuracy for full classification: 84.43717634232762%


Strategy 1